In [2]:
import math 
import time 

import spacy
import torch
import torchtext

from torch import nn, optim 
from torch.optim import Adam 
from torch import Tensor 

/data/anaconda3/envs/rosellm/lib/python3.10/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
print(device)

cpu


In [5]:
batch_size = 128  # How many sentences in a batch.
max_len = 256  # Maximum length of a sentence.

### Model parameters.

d_model = 512  # Word embedding size.
n_heads = 8  # Number of attention heads.

# Given d_model = 512 
# Given n_heads = 8 
# We have d_k = d_v = d_model / n_heads = 512 / 8 = 64
# which is the dimension of Q, K, V.

n_layers = 6  # Number of layers in the encoder/decoder.
d_ff = 2048  # Dimension of the feedforward network.

# 512 => 2048 => 512
# d_model => d_ff => d_model
# d_ff is also called "projection size".

n_hidden = d_ff  # Number of hidden units in the feedforward network.

dropout = 0.1  # Dropout probability.


### Optimizer parameters.

init_lr = 1e-5  # Initial learning rate.
factor = 0.9  # Factor by which the learning rate will be reduced.
adam_eps = 5e-9  # Adam epsilon.
patience = 10  # Patience for early stopping.
warmup = 100  # Number of steps for the warmup phase.
epoch = 100  # Number of epochs to train.
clip = 1.0  # Gradient clipping.
weight_decay = 5e-4  # Weight decay.
inf = float("inf")  # Infinity.


In [6]:
class Tokenizer:
    def __init__(self):
        self.spacy_de = spacy.load("de_core_news_sm")
        self.spacy_en = spacy.load("en_core_web_sm")

    def tokenize_de(self, text):
        return [tok.text for tok in self.spacy_de.tokenizer(text)]
    
    def tokenize_en(self, text):
        return [tok.text for tok in self.spacy_en.tokenizer(text)]
    
tokenizer = Tokenizer() 
example = 'This is an example sentence.'
tokens = tokenizer.tokenize_en(example)
print(example)
print(tokens)


/data/anaconda3/envs/rosellm/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'de_core_news_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/data/anaconda3/envs/rosellm/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


This is an example sentence.
['This', 'is', 'an', 'example', 'sentence', '.']


In [7]:
example = 'two yong, white males are outside near many bushes'
tokens = tokenizer.tokenize_de(example)
print(example)
print(tokens)


two yong, white males are outside near many bushes
['two', 'yong', ',', 'white', 'males', 'are', 'outside', 'near', 'many', 'bushes']


In [8]:
from torchtext.data import Field, BucketIterator
from torchtext.datasets import Multi30k
import urllib.request
import os
import gzip

class DataLoader:
    source: Field = None
    target: Field = None
    def __init__(self, ext, tokenize_en, tokenize_de, init_token, eos_token):
        self.ext = ext
        self.tokenize_en = tokenize_en
        self.tokenize_de = tokenize_de
        self.init_token = init_token
        self.eos_token = eos_token
        print(f"Initializing DataLoader with extension: {ext}")
        
    def download_dataset(self):
        """Download Multi30k dataset files from GitHub if they don't exist."""
        data_dir = '.data/multi30k'
        os.makedirs(data_dir, exist_ok=True)
        
        # Check if dataset files already exist.
        files = {
            'train': 'train',
            'val': 'val', 
            'test2016': 'test_2016_flickr'
        }
        
        all_files_exist = True
        for local_name in files.keys():
            for lang in ['en', 'de']:
                if not os.path.exists(f'{data_dir}/{local_name}.{lang}'):
                    all_files_exist = False
                    break
            if not all_files_exist:
                break
                
        if all_files_exist:
            print("Dataset files already exist. Skipping download.")
            return
            
        # Download and extract files if they don't exist.
        base_url = 'https://github.com/multi30k/dataset/raw/master/data/task1/raw'
        for local_name, remote_name in files.items():
            for lang in ['en', 'de']:
                final_path = f'{data_dir}/{local_name}.{lang}'
                if not os.path.exists(final_path):
                    url = f'{base_url}/{remote_name}.{lang}.gz'
                    compressed_path = f'{data_dir}/{local_name}.{lang}.gz'
                    
                    print(f'Downloading {url} to {compressed_path}')
                    urllib.request.urlretrieve(url, compressed_path)
                    
                    # Decompress .gz file
                    with gzip.open(compressed_path, 'rb') as f_in:
                        with open(final_path, 'wb') as f_out:
                            f_out.write(f_in.read())
                    
                    # Remove compressed file
                    os.remove(compressed_path)
    
    def make_dataset(self):
        # Download dataset files if needed
        self.download_dataset()
        
        if self.ext == ('.de', '.en'):
            self.source = Field(tokenize=self.tokenize_de, init_token=self.init_token, eos_token=self.eos_token, lower=True, batch_first=True)
            self.target = Field(tokenize=self.tokenize_en, init_token=self.init_token, eos_token=self.eos_token, lower=True, batch_first=True)
        elif self.ext == ('.en', '.de'):
            self.source = Field(tokenize=self.tokenize_en, init_token=self.init_token, eos_token=self.eos_token, lower=True, batch_first=True)
            self.target = Field(tokenize=self.tokenize_de, init_token=self.init_token, eos_token=self.eos_token, lower=True, batch_first=True)
        else:
            raise ValueError(f"Extension {self.ext} not supported.")
            
        train_data, valid_data, test_data = Multi30k.splits(
            exts=self.ext,
            fields=(self.source, self.target), 
            root='.data'
        )
        return train_data, valid_data, test_data
    
    def build_vocab(self, train_data, min_freq=2):
        self.source.build_vocab(train_data, min_freq=min_freq)
        self.target.build_vocab(train_data, min_freq=min_freq)

    def make_iter(self, train_data, valid_data, test_data, batch_size, device):
        train_iterator, valid_iterator, test_iterator = BucketIterator.splits(
            (train_data, valid_data, test_data),
            batch_size=batch_size,
            device=device
        )
        print(f"Train size: {len(train_iterator)}")
        print(f"Valid size: {len(valid_iterator)}")
        print(f"Test size: {len(test_iterator)}")
        print("Initialization done.")
        return train_iterator, valid_iterator, test_iterator

data_loader = DataLoader(
    ext=('.de', '.en'),
    tokenize_en=tokenizer.tokenize_en,
    tokenize_de=tokenizer.tokenize_de,
    init_token='<sos>',
    eos_token='<eos>'
)
train_data, valid_data, test_data = data_loader.make_dataset()
data_loader.build_vocab(train_data)
train_iterator, valid_iterator, test_iterator = data_loader.make_iter(train_data, valid_data, test_data, batch_size, device)

Initializing DataLoader with extension: ('.de', '.en')
Dataset files already exist. Skipping download.
Train size: 227
Valid size: 8
Test size: 8
Initialization done.


In [9]:
print(train_data.examples[0].src)
print(train_data.examples[0].trg)


['zwei', 'junge', 'weiße', 'männer', 'sind', 'im', 'freien', 'in', 'der', 'nähe', 'vieler', 'büsche', '.']
['two', 'young', ',', 'white', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


In [10]:
print("1. check vocabulary size")
print("source vocab size: ", len(data_loader.source.vocab))
print("target vocab size: ", len(data_loader.target.vocab))

1. check vocabulary size
source vocab size:  7853
target vocab size:  5893


In [11]:
print("2. check mapping between words and token indices")
print('word\t->\t token')
print("<sos>\t->\t ", data_loader.source.vocab.stoi['<sos>'])
print("<eos>\t->\t ", data_loader.source.vocab.stoi['<eos>'])
print("two\t->\t ", data_loader.target.vocab.stoi['two'])
print("many\t->\t ", data_loader.target.vocab.stoi['many'])
print("<pad>\t->\t ", data_loader.source.vocab.stoi['<pad>'])


2. check mapping between words and token indices
word	->	 token
<sos>	->	  2
<eos>	->	  3
two	->	  16
many	->	  202
<pad>	->	  1


In [12]:
print("3. get from iterator to select pairs")
# The function of <pad> is to pad the sentence to the same length.
# E.g. if the max length of sentence is l, then every sentence in the batch will be padded to l.
# My question: the max length of sentence is determined by the longest sentence in the training set or the max length of sentences in the batch?
# Answer: the max length of sentence is determined by the longest sentence in the batch.
for i, batch in enumerate(train_iterator):
    print(batch.src)
    print(batch.trg)
    break

3. get from iterator to select pairs
tensor([[  2,   5,  13,  ...,   1,   1,   1],
        [  2,   5, 522,  ...,   1,   1,   1],
        [  2,   5,  13,  ...,   1,   1,   1],
        ...,
        [  2,  54,  74,  ...,   1,   1,   1],
        [  2,   5,  13,  ...,   3,   1,   1],
        [  2,   8,  16,  ...,   1,   1,   1]])
tensor([[  2,   4,   9,  ...,   1,   1,   1],
        [  2,   4,  26,  ...,   1,   1,   1],
        [  2,   4, 174,  ...,   1,   1,   1],
        ...,
        [  2,  19, 152,  ...,   1,   1,   1],
        [  2,   4,   9,  ...,   3,   1,   1],
        [  2,   4,  14,  ...,   1,   1,   1]])


In [13]:
print("4. the following idices are also very important")
src_pad_idx = data_loader.source.vocab.stoi['<pad>']
trg_pad_idx = data_loader.target.vocab.stoi['<pad>']
src_sos_idx = data_loader.source.vocab.stoi['<sos>']
trg_sos_idx = data_loader.target.vocab.stoi['<sos>']
print(f"Source PAD index: {src_pad_idx}")
print(f"Target PAD index: {trg_pad_idx}")
print(f"Source SOS index: {src_sos_idx}")
print(f"Target SOS index: {trg_sos_idx}")

4. the following idices are also very important
Source PAD index: 1
Target PAD index: 1
Source SOS index: 2
Target SOS index: 2


In [14]:
enc_voc_size = len(data_loader.source.vocab)
dec_voc_size = len(data_loader.target.vocab)
print(f"Encoder vocabulary size: {enc_voc_size}, as the dimension of the input parameter x in the embedding layer")
print(f"Decoder vocabulary size: {dec_voc_size}, as the dimension of the output parameter x in the fully connected layer")

Encoder vocabulary size: 7853, as the dimension of the input parameter x in the embedding layer
Decoder vocabulary size: 5893, as the dimension of the output parameter x in the fully connected layer


In [15]:
# Get data from iter.
# only run once to ensure the same data is used for testing.
for i, batch in enumerate(test_iterator):
    src = batch.src
    trg = batch.trg
    print("save src shape: ", src.shape)
    print("save trg shape: ", trg.shape)
    torch.save(src, f".data/tensor_src.pt")
    torch.save(trg, f".data/tensor_trg.pt")
    print(src)
    print(trg)
    break

test_src = torch.load(".data/tensor_src.pt")
test_trg = torch.load(".data/tensor_trg.pt")
print("shape of test_src: ", test_src.shape)
print("shape of test_trg: ", test_trg.shape)


save src shape:  torch.Size([128, 10])
save trg shape:  torch.Size([128, 14])
tensor([[   2,   18, 6787,  ...,  123,    4,    3],
        [   2,  105,   41,  ...,   91,    4,    3],
        [   2,    5,   26,  ..., 3449,    4,    3],
        ...,
        [   2,    8,   16,  ...,    1,    1,    1],
        [   2,   26,   16,  ...,    1,    1,    1],
        [   2,   18,   30,  ...,    1,    1,    1]])
tensor([[   2,   16, 1909,  ...,    1,    1,    1],
        [   2,  110,   19,  ...,    1,    1,    1],
        [   2,    4,   34,  ...,    1,    1,    1],
        ...,
        [   2,    4,   14,  ...,    1,    1,    1],
        [   2,   24,   14,  ...,    1,    1,    1],
        [   2,   16,   30,  ...,    1,    1,    1]])
shape of test_src:  torch.Size([128, 10])
shape of test_trg:  torch.Size([128, 14])


/tmp/ipykernel_2342865/284749364.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_src = torch.load(".data/tensor_src.pt")
/tmp/ipykernel_2342865/284749364.py:15: Fu

In [16]:
import nltk
# BLEU: Bilingual Evaluation Understudy

def calculate_bleu(reference, candidate):
    reference = [reference.split()]
    candidate = candidate.split()
    smoothing_function = nltk.translate.bleu_score.SmoothingFunction()
    bleu = nltk.translate.bleu_score.sentence_bleu(reference, candidate, smoothing_function=smoothing_function.method1)
    return bleu

reference_sentence = "The cat is on the mat"
candidate_sentence = "The cat is on the mat"
bleu_score = calculate_bleu(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")
candidate_sentence = "The cat is setting on the mat"
bleu_score = calculate_bleu(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")

BLEU score: 1.0
BLEU score: 0.274941620352113


In [17]:
from collections import Counter
from fractions import Fraction
import math

def calculate_bleu_detailed(reference, candidate):
    """Calculate BLEU score between reference and candidate sentences."""
    print(f"original reference: {reference}, and original candidate: {candidate}.")
    # Convert strings to lists of words.
    reference = [reference.split()]  # Wrap in list since we expect list of references.
    candidate = candidate.split()
    print(f"reference: {reference}, and candidate: {candidate}.")
    
    # Use default weights for BLEU-4 (equal weights).
    weights = (0.25, 0.25, 0.25, 0.25)
    
    # Calculate modified n-gram precisions for n=1,2,3,4.
    precisions = []
    for n in range(1, 5):
        # Get n-grams for candidate and reference.
        candidate_ngrams = Counter([tuple(candidate[i:i+n]) 
                                  for i in range(len(candidate)-n+1)]) if len(candidate) >= n else Counter()
        print(f"n-gram: {n}, candidate n-grams: {candidate_ngrams}.")
        
        # Find max reference counts.
        max_ref_counts = {}
        for ref in reference:
            ref_ngrams = Counter([tuple(ref[i:i+n]) 
                                for i in range(len(ref)-n+1)]) if len(ref) >= n else Counter()
            for ngram in candidate_ngrams:
                max_ref_counts[ngram] = max(max_ref_counts.get(ngram, 0), ref_ngrams[ngram])
            print(f"reference n-grams: {ref_ngrams}.")
            print(f"max reference counts: {max_ref_counts}.")
        
        # Calculate clipped counts.
        clipped_counts = {ngram: min(count, max_ref_counts.get(ngram, 0))
                         for ngram, count in candidate_ngrams.items()}
        numerator = sum(clipped_counts.values())
        denominator = sum(candidate_ngrams.values()) if candidate_ngrams else 1
        precisions.append(0.1/denominator if not numerator else( 1.0*numerator / denominator if denominator else 0))
        print(f"clipped counts: {clipped_counts}, precision: {precisions[-1]}.")

    # Calculate brevity penalty.
    ref_len = len(reference[0])  # Use first reference length.
    cand_len = len(candidate)
    print(f"reference length: {ref_len}, candidate length: {cand_len}.")

    bp = math.exp(1 - ref_len/cand_len) if cand_len < ref_len else 1
    # Calculate final BLEU score.
    if precisions[0] == 0:
        return 0
    
    log_precisions = [math.log(p) if p > 0 else 0.1 for p in precisions]
    weighted_score = sum(w * p for w, p in zip(weights, log_precisions))
    bleu = bp * math.exp(weighted_score)
    print(f"brevity penalty: {bp}, log precisions: {log_precisions}, weighted score: {weighted_score}, BLEU score: {bleu}.")
    return bleu

In [18]:
reference_sentence = "The cat is on the mat"
candidate_sentence = "The cat is on the mat"
bleu_score = calculate_bleu_detailed(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")

original reference: The cat is on the mat, and original candidate: The cat is on the mat.
reference: [['The', 'cat', 'is', 'on', 'the', 'mat']], and candidate: ['The', 'cat', 'is', 'on', 'the', 'mat'].
n-gram: 1, candidate n-grams: Counter({('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}).
reference n-grams: Counter({('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}).
max reference counts: {('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}.
clipped counts: {('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}, precision: 1.0.
n-gram: 2, candidate n-grams: Counter({('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
reference n-grams: Counter({('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
max reference counts: {('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1

In [19]:
candidate_sentence = "The cat is setting on the mat"
bleu_score = calculate_bleu_detailed(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")

original reference: The cat is on the mat, and original candidate: The cat is setting on the mat.
reference: [['The', 'cat', 'is', 'on', 'the', 'mat']], and candidate: ['The', 'cat', 'is', 'setting', 'on', 'the', 'mat'].
n-gram: 1, candidate n-grams: Counter({('The',): 1, ('cat',): 1, ('is',): 1, ('setting',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}).
reference n-grams: Counter({('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}).
max reference counts: {('The',): 1, ('cat',): 1, ('is',): 1, ('setting',): 0, ('on',): 1, ('the',): 1, ('mat',): 1}.
clipped counts: {('The',): 1, ('cat',): 1, ('is',): 1, ('setting',): 0, ('on',): 1, ('the',): 1, ('mat',): 1}, precision: 0.8571428571428571.
n-gram: 2, candidate n-grams: Counter({('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'setting'): 1, ('setting', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
reference n-grams: Counter({('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).


In [20]:
from rouge import Rouge
from collections import Counter

# ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is a set of metrics used for evaluating automatic summarization and machine translation.
# It works by comparing overlapping units (n-grams, sequences, etc.) between a candidate text and reference text(s).

# Key metrics:
# - ROUGE-N: Measures overlap of n-grams between candidate and reference.
# - ROUGE-L: Measures longest common subsequence between candidate and reference.

# Key formulas:
# F1 score = 2 * (precision * recall) / (precision + recall).
# Precision = true_positives / (true_positives + false_positives).
# Recall = true_positives / (true_positives + false_negatives).

def calculate_rouge(reference, candidate):
    """
    Calculate ROUGE scores using both the Rouge package and custom implementation.
    
    Args:
        reference (str): The reference/ground truth text.
        candidate (str): The candidate/generated text to evaluate.
        
    Returns:
        tuple: ROUGE-1, ROUGE-2, and ROUGE-L F1 scores.
    """
    # Using the official Rouge package for verification.
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    rouge_1 = scores[0]['rouge-1']['f']  # Get F1 score for ROUGE-1.
    rouge_2 = scores[0]['rouge-2']['f']  # Get F1 score for ROUGE-2.
    rouge_l = scores[0]['rouge-l']['f']  # Get F1 score for ROUGE-L.
    
    # Calculate scores using our custom implementation.
    rouge_1_custom = calculate_rouge_n(reference, candidate, 1)
    rouge_2_custom = calculate_rouge_n(reference, candidate, 2)
    rouge_l_custom = calculate_rouge_l(reference, candidate)

    # Verify our implementation matches the official package.
    assert abs(rouge_1 - rouge_1_custom) < 1e-6, f"ROUGE-1 scores do not match, expected: {rouge_1}, actual: {rouge_1_custom}."
    assert abs(rouge_2 - rouge_2_custom) < 1e-6, f"ROUGE-2 scores do not match, expected: {rouge_2}, actual: {rouge_2_custom}."
    assert abs(rouge_l - rouge_l_custom) < 1e-6, f"ROUGE-L scores do not match, expected: {rouge_l}, actual: {rouge_l_custom}."
    
    print(f"Custom ROUGE scores - R1: {rouge_1_custom}, R2: {rouge_2_custom}, RL: {rouge_l_custom}.")
    return rouge_1, rouge_2, rouge_l

def calculate_rouge_n(reference, candidate, n):
    """
    Calculate ROUGE-N score for a given value of n.
    
    Args:
        reference (str): The reference text.
        candidate (str): The candidate text.
        n (int): The n-gram size to use.
        
    Returns:
        float: The ROUGE-N F1 score.

    Reference implementation from rouge/rouge_score.py:
    >>> def f_r_p_rouge_n(evaluated_count, reference_count, overlapping_count):
    >>>     # Handle edge case. This isn't mathematically correct, but it's good enough
    >>>     if evaluated_count == 0:
    >>>         precision = 0.0
    >>>     else:
    >>>         precision = overlapping_count / evaluated_count
    >>>     if reference_count == 0:
    >>>         recall = 0.0
    >>>     else:
    >>>         recall = overlapping_count / reference_count
    >>>     f1_score = 2.0 * ((precision * recall) / (precision + recall + 1e-8))
    >>>     return {"f": f1_score, "p": precision, "r": recall}
    """
    # Convert texts to lowercase and split into words.
    ref_words = reference.lower().split()
    cand_words = candidate.lower().split()
    
    # Create n-grams using sliding window and count their frequencies.
    ref_ngrams = Counter([tuple(ref_words[i:i+n]) for i in range(len(ref_words)-n+1)])
    cand_ngrams = Counter([tuple(cand_words[i:i+n]) for i in range(len(cand_words)-n+1)])
    print(f"reference n-grams: {ref_ngrams}.")
    print(f"candidate n-grams: {cand_ngrams}.")
    
    # Calculate number of matching n-grams using Counter intersection.
    matches = sum((ref_ngrams & cand_ngrams).values())
    
    if matches == 0:
        return 0
        
    # Calculate precision (matches / candidate n-grams) and recall (matches / reference n-grams).
    # The intuitive meaning of precision is the proportion of candidate n-grams that are also in the reference.
    # The intuitive meaning of recall is the proportion of reference n-grams that are also in the candidate.
    precision = matches / sum(cand_ngrams.values())
    recall = matches / sum(ref_ngrams.values())
    
    # F-measure or F-scores.
    # F_β = (1 + β²) * (precision * recall) / (β² * precision + recall)
    # When β = 1, F1 score = 2 * (precision * recall) / (precision + recall)
    # F1 score gives equal weight to precision and recall.
    # When β > 1, F_β favors recall more than precision.
    # When β < 1, F_β favors precision more than recall.
    # F1 score is most widely used. 
    # Because:
    # 1. It provides a single score that balances both precision and recall.
    # 2. It is particularly useful when you need a balance between precision and recall.
    # 3. It is the harmonic mean of precision and recall, which is more appropriate than arithmetic mean
    #    for rates and proportions.
    # Calculate F1 score using harmonic mean of precision and recall.
    if precision + recall == 0:
        return 0

    f1 = 2 * (precision * recall) / (precision + recall)
    
    return f1

def calculate_rouge_l(reference, candidate):
    """
    Calculate ROUGE-L score using Longest Common Subsequence (LCS).
    
    Args:
        reference (str): The reference text.
        candidate (str): The candidate text.
        
    Returns:
        float: The ROUGE-L F1 score.
    """
    # Convert texts to lowercase and split into words.
    ref_words = reference.lower().split()
    cand_words = candidate.lower().split()
    
    # Create DP table for LCS calculation.
    m, n = len(ref_words), len(cand_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    # Fill DP table using LCS algorithm.
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_words[i-1] == cand_words[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1  # Words match, extend LCS.
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])  # Take max of left or up.
    
    lcs_length = dp[m][n]  # Length of longest common subsequence.
    
    if lcs_length == 0:
        return 0
        
    # Calculate precision (LCS / candidate length) and recall (LCS / reference length).
    precision = lcs_length / len(cand_words)
    recall = lcs_length / len(ref_words)
    
    # Calculate F1 score.
    if precision + recall == 0:
        return 0
    f1 = 2 * (precision * recall) / (precision + recall)
    print(f"precision: {precision}, recall: {recall}, F1 score: {f1}.")
    return f1


In [21]:
# Why harmonic mean is more appropriate than arithmetic mean?
"""
The harmonic mean is more appropriate than arithmetic mean for rates and proportions for several key reasons:
* Better Handling of Extreme Values
* Penalizes Imbalance
* Properties for Rates
* Sensitivity to Low Values
The key advantages are:
Harmonic mean is always less than or equal to arithmetic mean.
It's more sensitive to low values in the data.
It better represents the "true" average when dealing with rates.
It naturally penalizes large imbalances between precision and recall.
This makes harmonic mean (F1 score) particularly suitable for evaluation metrics in machine learning where we want to:
Ensure balanced performance across different metrics.
Heavily penalize poor performance in any single metric.
Get a more realistic assessment of overall system performance.
"""

# Consider these precision-recall pairs
scores = [
    (1.0, 0.01),   # Very imbalanced
    (0.5, 0.5),    # Balanced
    (0.8, 0.2)     # Moderately imbalanced
]

for p, r in scores:
    arithmetic = (p + r) / 2
    harmonic = 2 * (p * r) / (p + r)
    print(f"P={p:.2f}, R={r:.2f}:")
    print(f"  Arithmetic mean: {arithmetic:.3f}")
    print(f"  Harmonic mean:   {harmonic:.3f}")

P=1.00, R=0.01:
  Arithmetic mean: 0.505
  Harmonic mean:   0.020
P=0.50, R=0.50:
  Arithmetic mean: 0.500
  Harmonic mean:   0.500
P=0.80, R=0.20:
  Arithmetic mean: 0.500
  Harmonic mean:   0.320


In [22]:
reference_sentence = "The cat is on the mat"
candidate_sentence = "The cat is on the mat"
rouge_1, rouge_2, rouge_l = calculate_rouge(reference_sentence, candidate_sentence)
print(f"ROUGE-1: {rouge_1}, ROUGE-2: {rouge_2}, ROUGE-L: {rouge_l}")

reference n-grams: Counter({('the',): 2, ('cat',): 1, ('is',): 1, ('on',): 1, ('mat',): 1}).
candidate n-grams: Counter({('the',): 2, ('cat',): 1, ('is',): 1, ('on',): 1, ('mat',): 1}).
reference n-grams: Counter({('the', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
candidate n-grams: Counter({('the', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
precision: 1.0, recall: 1.0, F1 score: 1.0.
Custom ROUGE scores - R1: 1.0, R2: 1.0, RL: 1.0.
ROUGE-1: 0.999999995, ROUGE-2: 0.999999995, ROUGE-L: 0.999999995


In [23]:
reference_sentence = "The cat is on the mat"
candidate_sentence = "The cat is setting on the mat"
rouge_1, rouge_2, rouge_l = calculate_rouge(reference_sentence, candidate_sentence)
print(f"ROUGE-1: {rouge_1}, ROUGE-2: {rouge_2}, ROUGE-L: {rouge_l}")

reference n-grams: Counter({('the',): 2, ('cat',): 1, ('is',): 1, ('on',): 1, ('mat',): 1}).
candidate n-grams: Counter({('the',): 2, ('cat',): 1, ('is',): 1, ('setting',): 1, ('on',): 1, ('mat',): 1}).
reference n-grams: Counter({('the', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
candidate n-grams: Counter({('the', 'cat'): 1, ('cat', 'is'): 1, ('is', 'setting'): 1, ('setting', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
precision: 0.8571428571428571, recall: 1.0, F1 score: 0.923076923076923.
Custom ROUGE scores - R1: 0.923076923076923, R2: 0.7272727272727272, RL: 0.923076923076923.
ROUGE-1: 0.9230769181065088, ROUGE-2: 0.7272727223140496, ROUGE-L: 0.9230769181065088


In [24]:
# The main purpose of the jiwer package is to provide implementations of various error 
# rate metrics used in speech recognition and text comparison tasks, with WER 
# being one of the most commonly used metrics. WER measures the minimum number 
# of word-level operations (insertions, deletions, substitutions) needed to 
# transform a reference text into a hypothesis text, normalized by the reference length.
import jiwer

def calculate_wer(reference, candidate):
    """
    Calculate Word Error Rate (WER) using both jiwer library and custom implementation.
    
    WER measures the minimum number of word-level operations (insertions, deletions, substitutions)
    needed to transform the reference text into the candidate text, normalized by reference length.
    
    Args:
        reference (str): The reference/ground truth text to compare against.
        candidate (str): The candidate/predicted text to evaluate.
        
    Returns:
        float: Word Error Rate score between 0.0 (perfect match) and >1.0 (very poor match).
    """
    # Calculate WER using the jiwer library for verification.
    jiwer_wer = jiwer.wer(reference, candidate)
    
    def custom_wer(ref, hyp):
        """
        Custom implementation of WER calculation using dynamic programming.
        
        Args:
            ref (str): Reference text.
            hyp (str): Hypothesis/candidate text.
            
        Returns:
            float: Custom calculated WER score.
        """
        # Split texts into word arrays for word-level comparison.
        ref_words = ref.split()
        hyp_words = hyp.split()
        
        # Initialize Levenshtein distance matrix with dimensions (ref_len + 1) x (hyp_len + 1).
        d = [[0] * (len(hyp_words) + 1) for _ in range(len(ref_words) + 1)]
        
        # Initialize first row and column with incremental distances.
        # These represent pure deletions and pure insertions respectively.
        for i in range(len(ref_words) + 1):
            d[i][0] = i  # Cost of deleting i words from reference.
        for j in range(len(hyp_words) + 1):
            d[0][j] = j  # Cost of inserting j words from hypothesis.
            
        # Fill the dynamic programming matrix.
        for i in range(1, len(ref_words) + 1):
            for j in range(1, len(hyp_words) + 1):
                if ref_words[i-1] == hyp_words[j-1]:
                    # If words match, no operation needed - take previous cost.
                    d[i][j] = d[i-1][j-1]
                else:
                    # Take minimum of three possible operations:
                    # - Substitution (diagonal)
                    # - Deletion (vertical)
                    # - Insertion (horizontal)
                    d[i][j] = min(d[i-1][j],    # Deletion.
                                d[i][j-1],      # Insertion.
                                d[i-1][j-1]     # Substitution.
                                ) + 1
                    
        # Calculate final WER by normalizing minimum edit distance.
        wer = d[-1][-1] / len(ref_words)
        return wer
    
    # Calculate WER using custom implementation.
    custom_wer_score = custom_wer(reference, candidate)
    
    # Verify both implementations produce equivalent results within floating point precision.
    assert abs(jiwer_wer - custom_wer_score) < 1e-10, \
        f"WER calculations don't match: jiwer={jiwer_wer} vs custom={custom_wer_score}."
    
    return jiwer_wer


In [25]:
reference_sentence = "The cat is on the mat"
candidate_sentence = "The cat is on the mat"
wer = calculate_wer(reference_sentence, candidate_sentence)
print(f"WER: {wer}")

WER: 0.0


In [26]:
reference_sentence = "The cat is on the mat"
candidate_sentence = "The cat is setting on the mat"
wer = calculate_wer(reference_sentence, candidate_sentence)
print(f"WER: {wer}")

WER: 0.16666666666666666


In [27]:
# Import required PyTorch modules.
import torch
import torch.nn.functional as F

# Word(string) => [token(int) -> vector(list(float))]

# Create an embedding layer with vocabulary size 14 and embedding dimension 512.
embd_layer = torch.nn.Embedding(14, 512)
print('Embedding weight shape:', embd_layer.weight.shape)

# Create input tensor with shape [batch_size=3, sequence_length=10].
# Each value represents a token ID in the vocabulary.
input_id = torch.tensor([[2, 4, 5, 6, 7, 8, 3, 1, 1, 1], 
                        [2, 4, 9, 10, 11, 12, 13, 3, 1, 1],
                        [2, 6, 7, 8, 9, 10, 11, 12, 13, 3]])

# Print shapes of input and embedded tensors.
print("Input tensor shape:", input_id.shape)
print("Embedded tensor shape:", embd_layer(input_id).shape)

# Print first 10 values of embedding vectors for position 1 in sequences 0 and 1.
print("First 10 values of embedding at position 1 in sequence 0:", embd_layer(input_id)[0][1][:10])
print("First 10 values of embedding at position 1 in sequence 1:", embd_layer(input_id)[1][1][:10])

Embedding weight shape: torch.Size([14, 512])
Input tensor shape: torch.Size([3, 10])
Embedded tensor shape: torch.Size([3, 10, 512])
First 10 values of embedding at position 1 in sequence 0: tensor([ 0.3226,  2.2697,  0.5828, -0.1000, -0.2319,  0.7776,  1.1028, -0.1409,
        -1.1775,  0.1281], grad_fn=<SliceBackward0>)
First 10 values of embedding at position 1 in sequence 1: tensor([ 0.3226,  2.2697,  0.5828, -0.1000, -0.2319,  0.7776,  1.1028, -0.1409,
        -1.1775,  0.1281], grad_fn=<SliceBackward0>)


In [28]:
# 1. Word2Vec using PyTorch built-in functions.
class Word2VecPyTorch(torch.nn.Module):
    """Word2Vec implementation using PyTorch built-in functions."""
    def __init__(self, vocab_size: int, embedding_dim: int):
        """Initialize Word2Vec model.
        
        Args:
            vocab_size: Size of vocabulary.
            embedding_dim: Dimension of word embeddings.
        """
        super().__init__()
        # Input word embeddings.
        self.in_embed = torch.nn.Embedding(vocab_size, embedding_dim)
        # Output word embeddings.
        self.out_embed = torch.nn.Embedding(vocab_size, embedding_dim)
        
    def forward(self, input_word: torch.Tensor, target_word: torch.Tensor) -> torch.Tensor:
        """Forward pass of Word2Vec model.
        
        Args:
            input_word: Input word indices tensor of shape (batch_size, seq_len).
            target_word: Target word indices tensor of shape (batch_size, seq_len).
            
        Returns:
            Loss value for the batch.
        """
        # Get input embeddings.
        input_embeds = self.in_embed(input_word)  # Shape: (batch_size, seq_len, embedding_dim).
        # Get output embeddings.
        output_embeds = self.out_embed(target_word)  # Shape: (batch_size, seq_len, embedding_dim).
        
        # Compute dot product between input and output embeddings.
        scores = torch.sum(input_embeds * output_embeds, dim=2)  # Shape: (batch_size, seq_len).
        
        # Apply sigmoid to get probabilities.
        return torch.sigmoid(scores)

# 2. Word2Vec from scratch.
class Word2VecScratch:
    """Word2Vec implementation from scratch."""
    def __init__(self, vocab_size: int, embedding_dim: int):
        """Initialize Word2Vec model.
        
        Args:
            vocab_size: Size of vocabulary.
            embedding_dim: Dimension of word embeddings.
        """
        # Initialize embeddings randomly.
        self.in_embed = torch.randn(vocab_size, embedding_dim, requires_grad=True)
        self.out_embed = torch.randn(vocab_size, embedding_dim, requires_grad=True)
        
    def forward(self, input_word: torch.Tensor, target_word: torch.Tensor) -> torch.Tensor:
        """Forward pass of Word2Vec model.
        
        Args:
            input_word: Input word indices tensor of shape (batch_size, seq_len).
            target_word: Target word indices tensor of shape (batch_size, seq_len).
            
        Returns:
            Loss value for the batch.
        """
        # Get input embeddings by indexing.
        input_embeds = self.in_embed[input_word]  # Shape: (batch_size, seq_len, embedding_dim).
        # Get output embeddings by indexing.
        output_embeds = self.out_embed[target_word]  # Shape: (batch_size, seq_len, embedding_dim).
        
        # Compute dot product manually.
        scores = torch.zeros(input_word.shape[0], input_word.shape[1])
        for i in range(input_word.shape[0]):
            for j in range(input_word.shape[1]):
                scores[i,j] = torch.sum(input_embeds[i,j] * output_embeds[i,j])
            
        # Apply sigmoid manually.
        return 1 / (1 + torch.exp(-scores))

# 3. Test to compare both implementations.
def test_word2vec_implementations():
    """Test and compare both Word2Vec implementations."""
    # Test parameters.
    vocab_size = 1000
    embedding_dim = 100
    batch_size = 32
    sentence_length = 10
    
    # Create random input data.
    input_words = torch.randint(0, vocab_size, (batch_size, sentence_length))
    target_words = torch.randint(0, vocab_size, (batch_size, sentence_length))
    
    # Initialize both models with same random seed.
    torch.manual_seed(42)
    pytorch_model = Word2VecPyTorch(vocab_size, embedding_dim)
    
    torch.manual_seed(42)
    scratch_model = Word2VecScratch(vocab_size, embedding_dim)
    
    # Copy weights from PyTorch model to scratch model.
    with torch.no_grad():
        scratch_model.in_embed = pytorch_model.in_embed.weight.clone()
        scratch_model.out_embed = pytorch_model.out_embed.weight.clone()
    
    # Get predictions from both models.
    pytorch_pred = pytorch_model(input_words, target_words)
    scratch_pred = scratch_model.forward(input_words, target_words)
    
    # Compare predictions.
    print("Maximum difference between predictions:", torch.max(torch.abs(pytorch_pred - scratch_pred)).item())
    print("Are predictions equal within tolerance?", torch.allclose(pytorch_pred, scratch_pred, atol=1e-6))

# Run the test.
test_word2vec_implementations()


Maximum difference between predictions: 1.1920928955078125e-07
Are predictions equal within tolerance? True


In [29]:
# http://jalammar.github.io/illustrated-word2vec/
# 1. Skip-gram implementation using PyTorch.
class SkipGramPyTorch(nn.Module):
    """Skip-gram model implementation using PyTorch modules."""
    
    def __init__(self, vocab_size: int, embedding_dim: int):
        """Initialize the Skip-gram model.
        
        Args:
            vocab_size: Size of the vocabulary.
            embedding_dim: Dimension of the word embeddings.
        """
        super().__init__()
        # Initialize input and output embeddings.
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)
        
    def forward(self, center_words: torch.Tensor, context_words: torch.Tensor) -> torch.Tensor:
        """Forward pass of Skip-gram model.
        
        Args:
            center_words: Tensor of shape (batch_size,) containing center word indices.
            context_words: Tensor of shape (batch_size, context_size) containing context word indices.
            
        Returns:
            Raw scores (logits) for each context word being predicted from the center word.
        """
        # Get embeddings for center and context words.
        center_embeds = self.in_embed(center_words).unsqueeze(1)  # (batch_size, 1, embed_dim)
        context_embeds = self.out_embed(context_words)  # (batch_size, context_size, embed_dim)
        
        # Compute dot product between center word and each context word.
        scores = torch.bmm(center_embeds, context_embeds.transpose(1, 2)).squeeze(1)  # (batch_size, context_size)
        
        return scores

# 2. Skip-gram implementation from scratch.
class SkipGramScratch:
    """Skip-gram model implementation from scratch."""
    
    def __init__(self, vocab_size: int, embedding_dim: int):
        """Initialize the Skip-gram model.
        
        Args:
            vocab_size: Size of the vocabulary.
            embedding_dim: Dimension of the word embeddings.
        """
        # Initialize embeddings randomly.
        self.in_embed = torch.randn(vocab_size, embedding_dim)
        self.out_embed = torch.randn(vocab_size, embedding_dim)
        
    def forward(self, center_words: torch.Tensor, context_words: torch.Tensor) -> torch.Tensor:
        """Forward pass of Skip-gram model.
        
        Args:
            center_words: Tensor of shape (batch_size,) containing center word indices.
            context_words: Tensor of shape (batch_size, context_size) containing context word indices.
            
        Returns:
            Raw scores (logits) for each context word being predicted from the center word.
        """
        batch_size = center_words.shape[0]
        context_size = context_words.shape[1]
        
        # Get embeddings for center words.
        center_embeds = self.in_embed[center_words]  # (batch_size, embed_dim)
        
        # Initialize scores tensor.
        scores = torch.zeros(batch_size, context_size)
        
        # For each example in batch.
        for i in range(batch_size):
            # For each context position.
            for j in range(context_size):
                # Get context word embedding.
                context_embed = self.out_embed[context_words[i,j]]
                # Compute dot product.
                scores[i,j] = torch.sum(center_embeds[i] * context_embed)
                
        return scores

# 3. Skip-gram implementation using gensim Word2Vec.
from gensim.models import Word2Vec
import numpy as np

class SkipGramGensim:
    """Skip-gram model implementation using gensim Word2Vec."""
    
    def __init__(self, vocab_size: int, embedding_dim: int):
        """Initialize the Skip-gram model using gensim.
        
        Args:
            vocab_size: Size of the vocabulary.
            embedding_dim: Dimension of the word embeddings.
        """
        # Create a dummy corpus to initialize the model.
        dummy_corpus = [[str(i) for i in range(vocab_size)]]
        
        # Initialize Word2Vec model.
        self.model = Word2Vec(
            sentences=dummy_corpus,
            vector_size=embedding_dim,
            window=5,  # Context window size
            min_count=1,
            workers=4,
            sg=1  # Skip-gram model
        )
        
    def forward(self, center_words: torch.Tensor, context_words: torch.Tensor) -> torch.Tensor:
        """Forward pass using gensim Word2Vec model.
        
        Args:
            center_words: Tensor of shape (batch_size,) containing center word indices.
            context_words: Tensor of shape (batch_size, context_size) containing context word indices.
            
        Returns:
            Raw scores (logits) for each context word being predicted from the center word.
        """
        batch_size = center_words.shape[0]
        context_size = context_words.shape[1]
        
        # Convert indices to strings for gensim.
        center_words_str = [str(idx.item()) for idx in center_words]
        context_words_str = [[str(idx.item()) for idx in context_words[i]] for i in range(batch_size)]
        
        # Initialize scores tensor.
        scores = torch.zeros(batch_size, context_size)
        
        # Compute similarity scores.
        for i in range(batch_size):
            center_vec = torch.tensor(self.model.wv[center_words_str[i]])
            for j in range(context_size):
                context_vec = torch.tensor(self.model.wv[context_words_str[i][j]])
                scores[i,j] = torch.dot(center_vec, context_vec)
                
        return scores

# 4. Test to compare all implementations.
def test_skipgram_implementations():
    """Test and compare all Skip-gram implementations."""
    # Test parameters.
    vocab_size = 1000
    embedding_dim = 100
    batch_size = 32
    context_size = 4
    
    # Create random input data.
    center_words = torch.randint(0, vocab_size, (batch_size,))
    context_words = torch.randint(0, vocab_size, (batch_size, context_size))
    
    # Initialize models with same random seed.
    torch.manual_seed(42)
    pytorch_model = SkipGramPyTorch(vocab_size, embedding_dim)
    
    torch.manual_seed(42)
    scratch_model = SkipGramScratch(vocab_size, embedding_dim)
    
    torch.manual_seed(42)
    gensim_model = SkipGramGensim(vocab_size, embedding_dim)
    
    # Copy weights from PyTorch model to scratch model.
    with torch.no_grad():
        scratch_model.in_embed = pytorch_model.in_embed.weight.clone()
        scratch_model.out_embed = pytorch_model.out_embed.weight.clone()
    
    # Get predictions from all models.
    pytorch_scores = pytorch_model(center_words, context_words)
    scratch_scores = scratch_model.forward(center_words, context_words)
    gensim_scores = gensim_model.forward(center_words, context_words)
    
    # Apply sigmoid to get probabilities.
    pytorch_pred = torch.sigmoid(pytorch_scores)
    scratch_pred = torch.sigmoid(scratch_scores)
    gensim_pred = torch.sigmoid(gensim_scores)
    
    # Compare predictions.
    print("PyTorch vs Scratch:")
    print("Maximum difference:", torch.max(torch.abs(pytorch_pred - scratch_pred)).item())
    print("Equal within tolerance?", torch.allclose(pytorch_pred, scratch_pred, atol=1e-6))
    
    print("\nPyTorch vs Gensim:")
    print("Maximum difference:", torch.max(torch.abs(pytorch_pred - gensim_pred)).item())
    print("Equal within tolerance?", torch.allclose(pytorch_pred, gensim_pred, atol=1e-6))

# Run the test.
test_skipgram_implementations()


PyTorch vs Scratch:
Maximum difference: 1.9371509552001953e-07
Equal within tolerance? True

PyTorch vs Gensim:
Maximum difference: 0.5001909732818604
Equal within tolerance? False
